# ReAct案例
本代码是一份实现ReAct的最小功能框架，主要为了帮助理解AI Agent最经典的工作方式，因此我没有添加其他skills等等。本案例的目标是了解ReAct的最基本循环结构和思维模式。

在正式看代码之前，先补充一些前置知识，如果你对AI Agent和ReAct基本概念已有了解则可跳过。

## 1. 什么是AI Agent？

AI Agent可以理解为一个能够自主感知环境、独立思考、并采取行动完成目标的程序。它就像一个有手有脚的AI大脑，不仅会思考还会执行。

普通的大模型问答，往往是用户提一个问题然后模型直接生成一个答案。

而AI Agent更强调过程而不是只强调结果。一个典型的Agent通常会做这些事情：
- 先理解目标是什么
- 再判断应该分几步完成
- 每一步决定下一步做什么
- 根据中间结果不断调整后续行为
- 最终给出答案或完成任务

所以，Agent的核心不只是会回答问题，而是会围绕目标持续推进任务。比如扫地机器人，如果是一般的遇到障碍物可能会卡住，但Agent会实时感受当前环境反馈来自主判断是否需要转弯、当前区域是否有异物、是否需要报告主人等等。

## 2. 什么是ReAct？

ReAct是AI Agent领域里最经典的基础框架之一，名字来自：
- `Reasoning`：推理、思考
- `Acting`：行动、执行下一步

它的核心思想很简单，就是先思考、再行动、观察结果、再继续思考......

也就是说，模型不是一上来就把最终答案一次性写完，而是把任务拆成若干轮循环。每一轮通常都包含：
- `Thought`：思考这一步
- `Action`：下一步要准备做什么
- `Observation`：这一步得到了什么结果

如果问题还没有解决，就继续进入下一轮；如果问题已经解决，就输出最终答案。

## 3. 为什么ReAct是主流？

ReAct框架的提出早在2022年（原论文地址：https://arxiv.org/abs/2210.03629
）。
但时至今日它仍然是许多智能体架构的主流框架。其有效性也得到广泛认可。这是因为很多真实任务并不是一步就能答完的。例如：复杂规划、解决复杂数学题、制定旅行方案、处理带有多个限制条件的问题等等。这类任务如果直接一次性生成最终答案，模型很容易漏条件、莫名其妙跳步、前后不一致、内部过程有问题等等

而ReAct之所以主流，是因为它有几个非常明显的优点：
- 结构清晰，容易看懂模型每一步在做什么
- 更适合多步任务，因为可以逐轮推进
- 更容易调试，因为可以看到是哪一轮出了问题
- 更容易扩展，以后可以很容易地接工具、数据库、记忆模块等等

你也可以把ReAct理解成很多复杂Agent系统的起点。哪怕后面加了工具调用、记忆、规划器、多Agent协作，最底层的思想通常仍然是分步思考、分步行动、根据结果继续推进。

因此，学习ReAct的意义不只是学会一个小框架，而是学会AI Agent最核心的一种工作方式。下面是本案例的完整代码。

In [1]:
#读取环境变量，我这里调用的是kimi模型，当然你可以选择openai的模型，
#只是openai的模型在国内访问可能会有各种各样的问题，而且有限额，价格也不是很友好。。。
#所以我的案例使用kimi。当然如果你有条件，也可以直接使用openai的模型，
#使用方法完全一样，只需要换一下model_name、base_url和api_key就行了。

import json
import os

from openai import OpenAI

#这里需要注意，个人API Key需要保密，不要泄露，可能会被人偷用，
#还有可能因此导致API Key作废（比如openai检测到开源代码里包含写明的API Key就会自动作废）。
#所以这里我们使用环境变量来存储API Key，不要写死成字符串，
#当然如果你仅仅在个人电脑上本地试验，那无所谓了~~~不过还是建议用环境变量。

#临时添加环境变量的powershell命令行：
'''
$env:MOONSHOT_API_KEY="<你的key>"
$env:MOONSHOT_BASE_URL="https://api.moonshot.cn/v1"
$env:KIMI_MODEL="kimi-k2.5"
'''

#如要永久添加环境变量：
'''
[Environment]::SetEnvironmentVariable("MOONSHOT_API_KEY", "<你的key>", "User")
[Environment]::SetEnvironmentVariable("MOONSHOT_BASE_URL", "https://api.moonshot.cn/v1", "User")
[Environment]::SetEnvironmentVariable("KIMI_MODEL", "kimi-k2.5", "User")
'''

api_key = os.getenv("MOONSHOT_API_KEY").strip()
api_key = api_key.strip('"').strip("'")

base_url = os.getenv("MOONSHOT_BASE_URL")
model_name = os.getenv("KIMI_MODEL")

#这里没有添加api_key的检查是因为如果环境变量没有api_key的话，上面就会直接出现None.strip()的报错
if not base_url:
    raise ValueError("请先在环境变量里配置MOONSHOT_BASE_URL")
if not model_name:
    raise ValueError("请先在环境变量里配置KIMI_MODEL")

client = OpenAI(api_key=api_key, base_url=base_url)

print("当前模型：", model_name)
print("当前base_url：", base_url)
print("成功读取环境变量")


当前模型： kimi-k2.5
当前base_url： https://api.moonshot.cn/v1
成功读取环境变量


## 1. 设置Prompt和具体问题
这里你可以：
- 直接修改`react_prompt`
- 直接修改`question`

你可以尝试不同的更复杂的question，看看大模型在ReAct模式下如何进行思考->推理->行动循环的。

In [2]:
react_prompt = """
你是一个用ReAct模式的AI Agent。
依靠已有知识，一步一步完成Thought->Action->Observation的循环。
要求：
1.Thought：写出这一步的简短思考；
2.Action：写出下一步准备做什么推理动作；
3.Observation：写出执行这一步后得到的中间结果；
4.循环多轮执行，每轮把大问题规划为多个小问题，每一轮完成一个小问题，直到你确信得到了最终答案，再给出final_answer。
你必须输出JSON对象，而且只能是下面两种格式之一：
{"thought": "你的简短思考", "action": "你的推理动作", "observation": "这一步得到的结果", "done": false}
或：
{"thought": "你的简短思考", "action": "你的推理动作", "observation": "这一步得到的结果", "done": true, "final_answer": "最终答案"}
注意：
1.不需要输出额外说明
2.每一轮要有对前一轮的自我检查，如果发现前一轮的action没有正确解决对应的小问题，要在下一轮的thought里指出来，并重新规划action
3.每一轮都必须包含thought、action、observation
""".strip()

#下面的question你可以自由发挥
question = "请帮我规划12月去火星游玩的攻略，包含行程安排、住宿推荐和美食推荐。"

print("当前问题：", question)


当前问题： 请帮我规划12月去火星游玩的攻略，包含行程安排、住宿推荐和美食推荐。


## 2. 把历史对话回喂给模型

ReAct的关键点之一是模型每一轮都要看到前面几轮已经做过什么，然后再决定下一步。我们的做法是通过Prompt Engineering保存对话历史，即每轮喂给大模型的prompt包括当前问题和对话历史。也有很多其他的Agent memory模式，包括RAG等等，实现起来更复杂一些，如果感兴趣可以自己查阅相关资料尝试一下。

In [3]:
def format_history(history):
    if not history:
        return "目前还没有历史步骤。"
    text = ""
    for i, step in enumerate(history, start=1):
        text += f"第 {i} 轮\n"
        text += f"Thought: {step['thought']}\n"
        text += f"Action: {step['action']}\n"
        text += f"Observation: {step['observation']}\n\n"
    return text.strip()

#下面是我们Prompt Engineering的核心，
#把用户问题和前面已经完成的历史进度一起输入给大模型，
#让大模型基于这些信息来规划下一步的Thought、Action和Observation。
def ask_llm(question, history, react_prompt):
    user_prompt = f"""
用户问题：
{question}

前面已经完成的步骤：
{format_history(history)}
""".strip()

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": react_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
    )

    text = response.choices[0].message.content.strip()

    return json.loads(text)


## 3. ReAct主循环

这部分是本案例的重点，因为它就是ReAct的核心：

- 先让模型给出这一轮的Thought/Action/Observation
- 把这一轮结果保存进历史里
- 再进入下一轮
- 如果`done = True`，代表模型认为已经得到最终结果了，就结束

In [4]:
def run_react(question, react_prompt, max_steps=10):
    history = []

    for step in range(1, max_steps + 1):
        print(f"\n========== 第 {step} 轮 ==========")

        result = ask_llm(question, history, react_prompt)

        thought = result["thought"]
        action = result["action"]
        observation = result["observation"]

        print("Thought:", thought)
        print("Action:", action)
        print("Observation:", observation)

        history.append({
            "thought": thought,
            "action": action,
            "observation": observation,
        })

        if result.get("done") is True:
            print("Final Answer:", result["final_answer"])
            return result["final_answer"]

    return "超过最大轮数还没得到最终答案。"


## 4. 输出结果，观察大模型使用的ReAct分析过程

你可以看到每一轮的：

- Thought
- Action
- Observation

最后还有：

- Final Answer

In [ ]:
final_answer = run_react(question, react_prompt)

print("\n最最最最最最最最最最终答案：", final_answer)



========== 第 1 轮 ==========
Thought: 用户要求规划12月去火星游玩的攻略。首先需要分析可行性：目前人类技术尚未实现商业化火星旅游，SpaceX等公司的火星计划仍在开发中，火星上没有现成的基础设施、住宿或餐厅。因此直接提供"攻略"是不现实的。我应该先确认当前火星旅行的技术现状，然后决定是提供基于未来预测的概念性攻略，还是说明现实限制。
Action: 确认当前火星旅游的技术可行性和基础设施现状
Observation: 目前火星旅游仍处于科幻和未来计划阶段。SpaceX的Starship正在开发中，NASA的载人火星任务计划在2030年代，目前还没有商业化的火星旅游线路。火星表面没有酒店、餐厅或旅游设施，环境极端（平均温度-60°C，大气稀薄，辐射强）。因此12月去火星游玩在现实中不可行。

========== 第 2 轮 ==========
Thought: 基于第1轮的观察，现实情况是火星旅游目前不可行。但用户明确要求攻略，因此我应该提供基于科学预测和未来技术的'概念性火星旅游攻略'，同时明确标注这是基于SpaceX、NASA等机构的火星计划预测（预计2030-2050年代可能实现）。需要设定一个合理的未来时间框架（如2040年代），并基于火星真实环境数据（12月是火星冬季，温度极低）来设计。
Action: 确定攻略框架：设定在2040年代初的首次商业化火星旅游，基于SpaceX Starship或类似技术，明确标注这是基于未来预测的概念性攻略，并考虑12月火星冬季的特殊环境。
Observation: 确定了攻略的基本设定：2040年代初，乘坐商业飞船前往火星，行程约6-9个月（去程）+ 停留数月 + 返程。12月对应火星北半球冬季（春分在12月左右，视具体年份），温度极低（-60°C至-125°C），需要特殊装备。攻略将基于未来火星基地（如Alpha Base或Mars City）的设想设施。

========== 第 3 轮 ==========
Thought: 基于前两轮设定的2040年代概念性火星旅游框架，现在需要具体规划行程安排。第2轮确定了12月是火星北半球冬季（春分前后），dust storm季节已过但温度极低。我需要将行程分解为：地球出发准备、星际航行、火星表面活动三个阶段，并特别针对12月冬季环境

: 